## Extraktion der täglichen Werte für NO2, den UV Aerosol Index und die meteorologischen ERA5-Daten an den Koordinaten für Würzburg.

**Einlesen der Sensor-Standorte um die Satellitendaten darauf zu kalibirieren**

In [17]:
import pandas as pd
import ee
from datetime import datetime, timedelta

# 1. Earth Engine Initialisierung
ee.Authenticate()
ee.Initialize(project='ee-arthurdreher01')

# 2. Parameter definieren
START_DATE = '2024-05-01'
END_DATE = '2024-08-31'

# 3. CSV einlesen und automatische FeatureCollection bauen
file_path = r"C:\Users\AD\Desktop\Uni Würzburg\EAGLE\InnoLab\Daten (Sensor_Community)\sensor_community_complete_may_august.csv"
df = pd.read_csv(file_path, sep=';', decimal=',')
stations_df = df[['sensor_id', 'lat', 'lon']].drop_duplicates().dropna()

ee_features = []
for index, row in stations_df.iterrows():
    try:
        lon_val = float(str(row['lon']).replace(',', '.'))
        lat_val = float(str(row['lat']).replace(',', '.'))
        geom = ee.Geometry.Point([lon_val, lat_val])
        feat = ee.Feature(geom, {'station_id': str(row['sensor_id'])})
        ee_features.append(feat)
    except ValueError:
        pass

# Das hier ersetzt deine alte stations_list!
stations_fc = ee.FeatureCollection(ee_features)

In [18]:
# 3. Qualitätsfilterung
def filter_no2(image):
    # Da qa_value fehlt, nutzen wir cloud_fraction (< 0.3 bedeutet weitestgehend wolkenfrei)
    cloud = image.select('cloud_fraction')
    mask = cloud.lt(0.5)
    return image.updateMask(mask).select('tropospheric_NO2_column_number_density')

def filter_uvai(image):
    # Das AER_AI Produkt in GEE besitzt keine Qualitäts- oder Wolkenbänder.
    # Wir wählen hier direkt den Aerosol-Index aus.
    return image.select('absorbing_aerosol_index')

In [19]:
# ---------------------------------------------------------
# 4. Tägliche Extraktionsfunktion (Mit separatem PBLH-Fix)
# ---------------------------------------------------------
def get_daily_data(date):
    current_date = ee.Date(date)
    next_date = current_date.advance(1, 'day')

    # 1. Satellitendaten abrufen (S5P)
    no2_col = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2") \
                .filterBounds(stations_fc) \
                .filterDate(current_date, next_date) \
                .map(filter_no2)

    uvai_col = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_AER_AI") \
                 .filterBounds(stations_fc) \
                 .filterDate(current_date, next_date) \
                 .map(filter_uvai)

    # 2. Meteorologie am Boden abrufen (ERA5-Land: 9 km Auflösung)
    era5_land_col = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY") \
                 .filterBounds(stations_fc) \
                 .filterDate(current_date, next_date) \
                 .select(['temperature_2m', 'u_component_of_wind_10m', 'v_component_of_wind_10m'])

    # 3. Atmosphärische Grenzschicht abrufen (ERA5 Global: 31 km Auflösung)
    era5_atm_col = ee.ImageCollection("ECMWF/ERA5/HOURLY") \
                 .filterBounds(stations_fc) \
                 .filterDate(current_date, next_date) \
                 .select(['boundary_layer_height'])

    # Tagesdurchschnitte bilden
    no2_daily = no2_col.mean()
    uvai_daily = uvai_col.mean()
    era5_land_daily = era5_land_col.mean()
    era5_atm_daily = era5_atm_col.mean()

    # Alles zu einem riesigen Multiband-Bild kombinieren
    combined_image = ee.Image.cat([no2_daily, uvai_daily, era5_land_daily, era5_atm_daily])

    # Punktgenaue Extraktion an allen Stationen gleichzeitig
    extracted_data = combined_image.reduceRegions(
        collection=stations_fc,
        reducer=ee.Reducer.first(),
        scale=1000
    )

    # Datum anheften
    def set_date(feature):
        return feature.set('date', current_date.format('YYYY-MM-dd'))
        
    return extracted_data.map(set_date)

In [20]:
# ---------------------------------------------------------
# 5. Iteration und Ausführung (Angepasst mit .flatten() )
# ---------------------------------------------------------
start_dt = datetime.strptime(START_DATE, '%Y-%m-%d')
end_dt = datetime.strptime(END_DATE, '%Y-%m-%d')
date_list = [(start_dt + timedelta(days=x)).strftime('%Y-%m-%d') for x in range((end_dt - start_dt).days + 1)]

# HIER IST DIE KORREKTUR: Wir nutzen nun len(stations_df)
print(f"Extrahiere Daten für {len(date_list)} Tage und {len(stations_df)} Stationen. Das kann einen Moment dauern...")

ee_dates = ee.List(date_list)

# Da wir jeden Tag eine Collection von Stationen zurückbekommen, 
# müssen wir die Listen am Ende mit .flatten() zu einer einzigen großen Tabelle zusammenlegen.
daily_features = ee.FeatureCollection(ee_dates.map(get_daily_data)).flatten()

Extrahiere Daten für 123 Tage und 8 Stationen. Das kann einen Moment dauern...


In [21]:
# ---------------------------------------------------------
# 6. Export als Background-Task an Google Drive (statt .getInfo)
# ---------------------------------------------------------
print("Bereite den Export-Task vor...")

# Wir filtern vorher leere Ergebnisse heraus, um die CSV sauber zu halten
clean_features = daily_features.filter(ee.Filter.notNull(['tropospheric_NO2_column_number_density']))

task = ee.batch.Export.table.toDrive(
    collection=clean_features,
    description='cloud0_5_Wuerzburg_S5P_ERA5_Extraction_2024',
    folder='GEE_Exports', # GEE erstellt diesen Ordner in deinem Google Drive
    fileFormat='CSV'
)

# Startet die Berechnung auf den Google-Servern
task.start()
print("Task erfolgreich gestartet! GEE rechnet nun im Hintergrund.")

# ---------------------------------------------------------
# 7. Status der Berechnung prüfen
# ---------------------------------------------------------
import time

# Eine kleine Schleife, die dir alle 30 Sekunden sagt, wie weit GEE ist
while task.active():
    print(f"Status: {task.status()['state']}... (Bitte warten)")
    time.sleep(30)

print(f"Fertig! Endgültiger Status: {task.status()['state']}")
print("Du kannst die fertige CSV-Datei nun aus deinem Google Drive (Ordner 'GEE_Exports') herunterladen.")

Bereite den Export-Task vor...
Task erfolgreich gestartet! GEE rechnet nun im Hintergrund.
Status: READY... (Bitte warten)
Fertig! Endgültiger Status: COMPLETED
Du kannst die fertige CSV-Datei nun aus deinem Google Drive (Ordner 'GEE_Exports') herunterladen.
